
# Batched walkers through a shared g

Run ``N`` walkers at once through *one* shared ``g`` with
:meth:`~flatwalk.WLDriver.run_batched`. Every tick is a single stacked call to
each batched callback — never a Python loop over walkers — so a vectorised
energy backend (NumPy here, but equally a GPU tensor or an MPI kernel) evaluates
all ``N`` configurations in one shot. All walkers scatter into the same ``g``,
so the single window converges faster than one walker would.

The batched contract mirrors the scalar one, one rank higher: each callback
takes an opaque ``state_batch`` of ``N`` walkers and returns ``N`` results. The
maths is in :doc:`/theory/07-multiple-walkers`.


## Block 1 — your batched physics

The batched Ising callbacks live in ``examples/ising_batched.py``. The state
is a single ``(N, L, L)`` int8 array — one array, not a list of lattices —
which is what lets the driver apply accepted moves with a boolean mask
``state[accept] = new_state[accept]``.



In [ ]:
import sys

try:
    from pathlib import Path

    sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "examples"))
except NameError:
    pass  # sphinx-gallery exec context: __file__ undefined, sys.path is already set

import beale  # noqa: E402
import ising  # noqa: E402
import ising_batched  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402

from flatwalk import Bin1D, WLConfig, WLDriver  # noqa: E402

L = 4
N = 8  # walkers in the shared window

cb = ising_batched.make_batched_ising_callbacks(L)
low, high, n_bins = ising.ising_energy_bins(L)
scheme = Bin1D(low, high, n_bins)

rng = np.random.default_rng(0)
initial_state = rng.choice(np.array([-1, 1], dtype=np.int8), size=(N, L, L))

## Block 2 — generic wiring, via run_batched

``t_total`` counts individual moves (``N`` per tick), so ``n_check``,
``ln_f_final`` and the 1/t schedule use the same units as the scalar
:meth:`~flatwalk.WLDriver.run`; with ``n_walkers=1`` the bookkeeping reduces
to the scalar schedule exactly.



In [ ]:
cfg = WLConfig(bin_scheme=scheme, beta=0.0, n_check=2_000, ln_f_final=1e-5)
result = WLDriver(cfg).run_batched(
    initial_state=initial_state,
    energy_fn=cb["energy_fn"],
    order_parameter_fn=cb["order_parameter_fn"],
    propose_move_fn=cb["propose_move_fn"],
    n_walkers=N,
    rng=rng,
)
print(
    f"{N} walkers, {result.t_total:,} moves, {result.n_f_stages} f-stages, "
    f"converged={result.converged}"
)

## Compare the shared g against Beale



In [ ]:
log_g_exact = beale.log_g_E_array(L, beale.beale_g_E(L), scheme.centers)
n_E_exact = np.exp(log_g_exact)
valid = result.visited & np.isfinite(log_g_exact) & (n_E_exact > 0)

n_E_WL = np.exp(result.g[valid] - result.g[valid].max())
n_E_WL *= n_E_exact[valid].sum() / n_E_WL.sum()
eps = np.abs(n_E_WL - n_E_exact[valid]) / n_E_exact[valid]
central = np.ones_like(eps, dtype=bool)
central[0] = central[-1] = False
print(f"  max ε = {eps[central].max():.3f}, mean ε = {eps[central].mean():.3f}")
assert eps[central].mean() < 0.2, "smoke batched mean ε too large"

E_axis = scheme.centers[valid]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(
    E_axis,
    result.g[valid] - result.g[valid].min(),
    "o-",
    color="C0",
    label=f"batched WL ({N} walkers)",
)
ax.plot(
    E_axis,
    log_g_exact[valid] - log_g_exact[valid].min(),
    "k--",
    lw=1.0,
    label="Beale exact",
)
ax.set_xlabel("E")
ax.set_ylabel("log g(E)   (shifted to min = 0)")
ax.set_title(f"L={L} Ising: {N} batched walkers vs Beale")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

To wire your own batched backend, supply callbacks that take a stacked
``state_batch`` and return ``ndarray[N]`` (energy, order parameter) or
``(new_state_batch, lpr[N])`` (proposal). Distinct windows with per-window
``g`` and exchange are :doc:`replica exchange <plot_5_replica_exchange_ising>`.

